<a href="https://colab.research.google.com/github/BenFreilichUVA/ds2002-fa26/blob/main/2026_09_25_%E2%80%94_Cleaning_Gauntlet_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
_price_check = df['price'].str.strip('$').astype(float)
revenue_before_step_1 = round((df['qty'] * _price_check).sum(), 2)

removed = int(df.duplicated().sum())  # TODO: how many duplicates were there?
df = df.drop_duplicates().copy()  # TODO: df with duplicates dropped, copied
log('duplicates', 'dropped exact duplicate rows', removed)

_price_check = df['price'].str.strip('$').astype(float)
revenue_after_step_1 = round((df['qty'] * _price_check).sum(), 2)

[duplicates] dropped exact duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
df['price'] = df['price'].str.strip('$').astype(float)

assert df['price'].dtype == float
log('price', 'stripped the dollar sign and converted the type to a float (column arrived as string)', len(df))

[price] stripped the dollar sign and converted the type to a float (column arrived as string) (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

before_drop = len(df)
df = df[df['qty'] >= 1].copy()   # drops NaN too, since NaN >= 1 is False
removed_qty = before_drop - len(df)
revenue_after_step_3 = round((df['qty'] * df['price']).sum(), 2)
log('qty', 'coerced qty to numeric; dropped rows with missing or non-positive qty', removed_qty)

[qty] coerced qty to numeric; dropped rows with missing or non-positive qty (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
print(df['item'].value_counts())
ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho',
}
df['item'] = df['item'].map(ITEM_MAP)
print(df['item'].value_counts())
log('item', 'canonicalized item names', len(df))

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64
[item] canonicalized item names (275 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
print(df['category'].value_counts())
ITEM_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'Merch': 'Merch',
    'Apparel': 'Merch',
    'RainGear': 'RainGear',
    'rain-gear': 'RainGear',
}
df['category'] = df['category'].map(ITEM_MAP)
print(df['category'].value_counts())
log('category', 'canonicalized category names', len(df))

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
category
Food        95
Merch       94
RainGear    86
Name: count, dtype: int64
[category] canonicalized category names (275 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert df['item'].nunique() == 3
assert df['category'].nunique() == 3
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
df['revenue'] = df['qty'] * df['price']

by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2)
print(by_category)

total_revenue = round(df['revenue'].sum(), 2)
print('Total revenue:', total_revenue)

category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64
Total revenue: 4740.0


**What I would tell the vendor:** I would tell the vendor to stock more merch and food. Even if it was pouring rain, I would still expect fans to have brought their own rain gear. The fact that rain gear is comparable in sales with merch and food is strange.



### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,dropped exact duplicate rows,15
1,price,stripped the dollar sign and converted the typ...,300
2,qty,coerced qty to numeric; dropped rows with miss...,25
3,item,canonicalized item names,275
4,category,canonicalized category names,275


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

In [11]:
print(revenue_before_step_1)
print(revenue_after_step_1)
print(revenue_after_step_3)

4852.5
4594.5
4740.0


A. Taking out the duplicates changed the revenue the most. It changed the revenue by $258, a drop larger than 5%. This happened because a lot of duplicate transactions were removed from the dataset, which gives us the actual revenue number, not the inflated number.

B. Someone else may have decided not to lump Apparel and Merch together in the category selection, whereas I decided to merge the two. It would not have changed overall reported revenue, however it would have changed the category by revenue substantially, where rain gear would be ahead both merch and apparel. I went the way I did because I reasoned that Apparel is just a subgenre of merch, and by keeping them separate someone who just looks at the dataset may not properly realize how much merch and clothing sales actually contributed to the revenue top line.